# Principal Component Analysis

So far every model used `X` and `y`. Here there is no `y`.

PCA takes a data set with many features and builds a few new features that keep most
of the differences between the points. We use it to plot high-dimensional data and to
shrink `X` before a model.

## 1. Why reduce dimensions?

The digits data has 64 features. One feature per pixel.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits

digits = load_digits()
X_digits = digits.data
y_digits = digits.target

print("shape:", X_digits.shape)

We can draw a scatter plot of 2 features. We cannot draw one of 64.

We also expect repeats: pixels in the corner are blank in almost every image, and
neighbouring pixels usually look alike. 64 numbers carry less than 64 numbers' worth
of information.

## 2. PCA in sklearn

On the worksheet you built both components of this data by hand: centre the columns,
project onto a direction, and keep the direction that leaves the values most spread
out.

Here is the same data, and sklearn doing the same job in one line. It centres the
columns and finds the directions for you.

In [ ]:
from sklearn.decomposition import PCA

# the five students from the worksheet: hours studied, practice problems
X = np.array([[10, 19],
              [ 2, 25],
              [12, 30],
              [22, 35],
              [14, 41]], dtype=float)

pca = PCA(n_components=2)
scores_2d = pca.fit_transform(X)

print("components (one per row):")
print(pca.components_)
print()
print("explained variance ratio:", pca.explained_variance_ratio_)
print()
print("the five points in the new coordinates:")
print(scores_2d)

Three things to read off:

- `components_[0]` is `(0.6, 0.8)`, the direction we found by hand.
- `explained_variance_ratio_` is `[0.8, 0.2]`. PC1 holds 80% of the spread, PC2 holds 20%.
- The new coordinates are the projections. Column 1 is the PC1 score of each student.

Keeping only column 1 throws away 20% of the spread and one of the two features.

## 3. Digits: how many components?

Fit PCA on all 64 features and look at how the spread is split.

In [ ]:
pca_d = PCA().fit(X_digits)
ratio = pca_d.explained_variance_ratio_
cumulative = np.cumsum(ratio)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].bar(range(1, 21), ratio[:20])
ax[0].set_xlabel("component")
ax[0].set_ylabel("variance ratio")
ax[0].set_title("first 20 components")
ax[1].plot(range(1, 65), cumulative, marker=".")
ax[1].axhline(0.9, color="C1", linestyle="--")
ax[1].set_xlabel("number of components")
ax[1].set_ylabel("cumulative ratio")
ax[1].set_title("running total")
for a in ax:
    a.grid(alpha=0.3)
plt.tight_layout()
plt.show()

for target in [0.8, 0.9, 0.95]:
    k = int(np.searchsorted(cumulative, target) + 1)
    print(f"{target:.0%} of the spread needs {k} components")

21 components hold 90% of the spread. The other 43 hold 10%.

The first plot is called a scree plot. The drop after the first few components is the
usual reason to stop there.

## 4. Digits in two dimensions

Now we can plot 64-dimensional data.

In [ ]:
XY = PCA(n_components=2).fit_transform(X_digits)

plt.figure(figsize=(7, 6))
sc = plt.scatter(XY[:, 0], XY[:, 1], c=y_digits, cmap="tab10", s=12)
plt.colorbar(sc, label="digit")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.grid(alpha=0.3)
plt.show()

Those two components hold 28.5% of the spread, and the digits already
separate into groups. Nothing in the fit used `y`; the colours were added afterwards.

**Discussion.** Which digits land on top of each other, and what would you guess that
says about the images?

## 5. What is a component?

A component is a weight for every original feature. Here that is a weight per pixel,
so we can draw it as an 8x8 image.

In [ ]:
fig, axes = plt.subplots(1, 6, figsize=(11, 2.2))
for i, a in enumerate(axes):
    a.imshow(pca_d.components_[i].reshape(8, 8), cmap="RdBu")
    a.set_title(f"PC{i + 1}")
    a.axis("off")
plt.show()

Red and blue are opposite signs. Each component is a pattern of light
and dark that many digits share.

Every image is rebuilt as the mean image plus a weighted sum of these patterns. Use
more components and you get a closer copy.

In [ ]:
image = X_digits[0]
ks = [1, 5, 20, 40, 64]

fig, axes = plt.subplots(1, len(ks) + 1, figsize=(11, 2.2))
axes[0].imshow(image.reshape(8, 8), cmap="gray")
axes[0].set_title("original")
axes[0].axis("off")

for a, k in zip(axes[1:], ks):
    p = PCA(n_components=k).fit(X_digits)
    back = p.inverse_transform(p.transform(image.reshape(1, -1)))
    a.imshow(back.reshape(8, 8), cmap="gray")
    a.set_title(f"k = {k}")
    a.axis("off")
plt.show()

## 6. Does PCA hurt the model?

Fit logistic regression on all 64 features, then on the first `k` components.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X_tr, X_te, y_tr, y_te = train_test_split(
    X_digits, y_digits, test_size=0.3, random_state=0, stratify=y_digits)

full = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))
full.fit(X_tr, y_tr)
print(f"all 64 features   test accuracy {full.score(X_te, y_te):.3f}")
print()

for k in [2, 5, 10, 20, 40]:
    model = make_pipeline(StandardScaler(),
                          PCA(n_components=k, random_state=0),
                          LogisticRegression(max_iter=5000))
    model.fit(X_tr, y_tr)
    print(f"k = {k:2d} components   test accuracy {model.score(X_te, y_te):.3f}")

20 components get close to all 64. Two do not.

**Discussion.** PCA never looked at `y`. Why can it still keep the information the
classifier needs?

## 7. Scale first

PCA chases spread, and spread depends on units. A feature measured in large numbers
will win no matter what it means.

The wine data has 13 chemical measurements on very different scales.

In [ ]:
from sklearn.datasets import load_wine

wine = load_wine()
X_wine, y_wine = wine.data, wine.target

for name, v in sorted(zip(wine.feature_names, X_wine.var(axis=0)),
                      key=lambda t: -t[1])[:3]:
    print(f"{name:<22} variance {v:12.2f}")

In [ ]:
np.set_printoptions(suppress=True)

raw = PCA(n_components=3).fit(X_wine)
print("unscaled ratios:", np.round(raw.explained_variance_ratio_, 4))

top = np.argsort(np.abs(raw.components_[0]))[::-1][:3]
print("\nPC1 weights:")
for i in top:
    print(f"  {wine.feature_names[i]:<22} {raw.components_[0][i]: .4f}")

PC1 holds 99.8% of the spread and is almost entirely one feature,
`proline`. That is not structure, it is a unit. Proline is measured in the hundreds
and everything else is not.

Scale the columns first and refit.

In [ ]:
X_wine_s = StandardScaler().fit_transform(X_wine)
scaled = PCA(n_components=5).fit(X_wine_s)

print("scaled ratios:    ", np.round(scaled.explained_variance_ratio_, 4))
print("running total:    ", np.round(np.cumsum(scaled.explained_variance_ratio_), 4))

Now the spread is shared out and PC1 is a real mix of measurements.

Rule: scale before PCA unless every feature is already in the same unit. The digits
pixels all run 0 to 16, which is why we did not scale them.

## Your Turn: the wine data

The wine data has 178 bottles, 13 measurements, 3 growers (`y_wine`). Work with the
scaled version, `X_wine_s`.

Answer the questions in a text cell under each step.

### Step 1

Fit PCA with all components. Plot the scree plot and the running total.
How many components do you need for 90% of the spread?

In [ ]:
# TODO: fit PCA() on X_wine_s
# TODO: plot the ratios and the running total
# TODO: print how many components reach 80%, 90%, 95%


### Step 2

Project the data onto the first two components and plot it, coloured by `y_wine`.
Do the three growers separate?

In [ ]:
# TODO: transform X_wine_s to 2 components
# TODO: scatter plot coloured by y_wine


### Step 3

Print the 4 largest weights in PC1. What do those measurements have in common?

In [ ]:
# TODO: get components_[0] from a PCA fit on X_wine_s
# TODO: print the 4 features with the largest weights


### Step 4

Split the data. Fit logistic regression on all 13 scaled features, then on 2, 3 and 5
components. Compare the test accuracy.

In [ ]:
# TODO: train_test_split on X_wine, y_wine (test_size=0.3, random_state=0, stratify=y_wine)
# TODO: pipeline StandardScaler + LogisticRegression on all 13
# TODO: same with PCA(n_components=k) for k = 2, 3, 5


### Step 5

Which model would you hand in, and why? Write two or three sentences.

---

## What we did not cover

PCA only finds straight directions. If the structure in your data curves, PCA will
miss it. The circles data set from the neural network notebook is the clear case:
no straight direction separates the two rings.